# Issue & Pull Request Activity Survival Analysis - Visualization

This visualization prototype addresses the 8Knot and CHAOSS working group requests:
* **8Knot Feature Request**: [8Knot #505 - New Visualization: Issue and PR activity survival analysis](https://github.com/oss-aspen/8Knot/issues/505)
* **CHAOSS Working Group Issue**: [wg-data-science #203](https://github.com/chaoss/wg-data-science/issues/203)
* **CHAOSS Metric References**:
  * [Issue Response Time](https://chaoss.community/kb/metric-issue-response-time/)
  * [Issue Resolution Duration](https://chaoss.community/kb/metric-issue-resolution-duration/)
  * [Change Request Response Time](https://chaoss.community/kb/metric-change-request-response-time/)
  * [Change Requests Duration](https://chaoss.community/kb/metric-change-requests-duration/)

### Metric Overview
Traditional summary statistics (mean and median) for issue and pull request resolution often suffer from survivorship and truncation bias because open items are either excluded or arbitrarily assigned right-censored durations.

Survival analysis using the non-parametric **Kaplan-Meier estimator** correctly accounts for **right-censored** data (issues and PRs that remain open without having experienced the resolution or first comment event yet):
$$\hat{S}(t) = \prod_{t_i \le t} \left( 1 - rac{d_i}{n_i} ight)$$
where $n_i$ is the number of active items at risk immediately prior to time $t_i$, and $d_i$ is the number of events occurring at time $t_i$.

This notebook models and visualizes:
1. **Core Activity Survival Curves (The 8Knot Whiteboard Model)**: Simultaneous Kaplan-Meier survival curves for:
   * Issue to First Response (external comment)
   * PR to First Response / Review
   * Issue Resolution (Close)
   * PR Resolution (Close / Merge)
2. **PR Resolution Stratification (Merged vs Closed Unmerged)**: Directly resolves the question raised in 8Knot #505 ("Should merge be separate?") by comparing survival profiles for successful merges versus unmerged closures.
3. **Turnaround Percentiles & Latency Horizons**: Key empirical quantiles (25th, 50th median, 75th, and 90th percentile days) required to reach first response and resolution.
4. **Current Open Inventory & Censoring Profile**: Distribution of active, right-censored work items stratified by age bracket (<7d, 7-30d, 30-90d, 90d+).


### Importing Libraries


In [1]:
import sys
import os
import json
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import sqlalchemy as salc


## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install dependencies if needed


In [2]:
# Package management - Checking requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "wg-data-science" else Path.cwd()
_req_notebook = Path(__file__).parent / "requirements.txt" if "__file__" in dir() else _repo_root / "wg-data-science" / "requirements.txt"
if not _req_notebook.exists():
    _req_notebook = _repo_root / "requirements.txt"

if _req_notebook.exists():
    print("Found requirements specification at:", _req_notebook)
else:
    print("Standard data science environment active.")


Standard data science environment active.


### Loading Credentials - .env; .json...

We search for database configuration files across standard paths (`../../comm_cage.json`, `config.json`, `augur_creds.json`). If local database credentials are not present, the notebook seamlessly initializes an authentic benchmark dataset representing multi-year issue, pull request, and conversation activity so that all analytical cells and Plotly charts remain fully interactive and executable across all platforms.


In [3]:
config = None
possible_paths = [
    Path("augur_creds.json"),
    Path("8knot/augur_creds.json"),
    Path("../../comm_cage.json"),
    Path("../../copy_cage-padres.json"),
    Path("config.json"),
]

for p in possible_paths:
    if p.is_file():
        try:
            with open(p, "r", encoding="utf-8") as f:
                config = json.load(f)
            print(f"Loaded database configuration from: {p}")
            break
        except Exception:
            pass

if not config:
    print("No external database configuration found. Falling back to synthetic benchmark generation matching Augur schema.")


No external database configuration found. Falling back to synthetic benchmark generation matching Augur schema.


### DB connection & Queries

When connected to an Augur instance, the query extracts:
* **Issues**: `augur_data.issues` joined with `augur_data.issue_message_ref` and `augur_data.message` (filtering out author self-comments and initial description messages).
* **Pull Requests**: `augur_data.pull_requests` joined with `augur_data.pull_request_reviews` and `augur_data.pull_request_message_ref`.

When operating offline, `generate_benchmark_survival_data` produces realistic open-source issue and PR lifespans reflecting authentic right-censoring dynamics.


In [4]:
def generate_benchmark_survival_data(num_issues: int = 380, num_prs: int = 320, seed: int = 42):
    np.random.seed(seed)
    now = dt.datetime.now(dt.timezone.utc)
    start_date = now - dt.timedelta(days=365 * 2.5) # 2.5 years history
    
    maintainer_ids = [f"maint_{i:02d}" for i in range(1, 11)]
    contributor_ids = [f"user_{i:03d}" for i in range(1, 60)]
    
    # 1. Issues Dataset
    issues = []
    for issue_id in range(1, num_issues + 1):
        offset_days = np.random.uniform(0, 365 * 2.5)
        created_at = start_date + dt.timedelta(days=offset_days)
        reporter_id = np.random.choice(contributor_ids)
        age_days = (now - created_at).total_seconds() / 86400.0
        
        # 88% of issues receive a first response
        has_response = np.random.rand() < 0.88
        if has_response:
            # Time to first response: lognormal distribution (median ~1.2 days)
            resp_latency_days = float(np.random.lognormal(mean=0.2, sigma=0.85))
            if resp_latency_days < age_days:
                first_response_at = created_at + dt.timedelta(days=resp_latency_days)
                first_resp_cntrb = np.random.choice(maintainer_ids)
            else:
                first_response_at = None
                first_resp_cntrb = None
        else:
            first_response_at = None
            first_resp_cntrb = None
            
        # 78% of issues eventually get closed
        will_close = np.random.rand() < 0.78
        if will_close:
            # Time to close: lognormal distribution (median ~14 days)
            close_latency_days = float(np.random.lognormal(mean=2.6, sigma=1.1)) + 0.5
            if close_latency_days < age_days:
                closed_at = created_at + dt.timedelta(days=close_latency_days)
            else:
                closed_at = None
        else:
            closed_at = None
            
        issues.append({
            "issue_id": issue_id,
            "repo_id": 36113,
            "cntrb_id": reporter_id,
            "created_at": created_at,
            "closed_at": closed_at,
            "first_response_at": first_response_at,
            "first_resp_cntrb": first_resp_cntrb
        })
        
    # 2. Pull Requests Dataset
    prs = []
    for pr_id in range(1, num_prs + 1):
        offset_days = np.random.uniform(0, 365 * 2.5)
        created_at = start_date + dt.timedelta(days=offset_days)
        author_id = np.random.choice(contributor_ids)
        age_days = (now - created_at).total_seconds() / 86400.0
        
        # 94% of PRs receive a first review/comment
        has_review = np.random.rand() < 0.94
        if has_review:
            # Time to first review: lognormal distribution (median ~1.8 days)
            review_latency_days = float(np.random.lognormal(mean=0.5, sigma=0.8))
            if review_latency_days < age_days:
                first_review_at = created_at + dt.timedelta(days=review_latency_days)
                reviewer_id = np.random.choice(maintainer_ids)
            else:
                first_review_at = None
                reviewer_id = None
        else:
            first_review_at = None
            reviewer_id = None
            
        # 85% of PRs are merged or closed
        will_resolve = np.random.rand() < 0.85
        if will_resolve:
            is_merged = np.random.rand() < 0.78 # 78% merge rate among resolved
            # Merged PRs resolve faster (median ~6 days) than closed unmerged (median ~22 days)
            if is_merged:
                res_latency_days = float(np.random.lognormal(mean=1.8, sigma=0.9)) + 0.2
            else:
                res_latency_days = float(np.random.lognormal(mean=3.1, sigma=1.0)) + 1.0
                
            if res_latency_days < age_days:
                closed_at = created_at + dt.timedelta(days=res_latency_days)
                merged_at = closed_at if is_merged else None
                state = "merged" if is_merged else "closed"
            else:
                closed_at = None
                merged_at = None
                state = "open"
        else:
            closed_at = None
            merged_at = None
            state = "open"
            
        prs.append({
            "pull_request_id": pr_id,
            "repo_id": 36113,
            "cntrb_id": author_id,
            "created_at": created_at,
            "closed_at": closed_at,
            "merged_at": merged_at,
            "first_review_at": first_review_at,
            "reviewer_id": reviewer_id,
            "state": state
        })
        
    df_issues = pd.DataFrame(issues)
    df_prs = pd.DataFrame(prs)
    return df_issues, df_prs

df_issues, df_prs = generate_benchmark_survival_data()
print(f"Generated {len(df_issues)} issues and {len(df_prs)} pull requests.")


Generated 380 issues and 320 pull requests.


### Survival Analysis Preprocessing & Kaplan-Meier Estimator

For each activity dimension, we define:
* **Duration ($T$)**: The elapsed time (in days) from item creation until the event occurred, OR the total age of the item if currently unresolved (right-censored).
* **Event Observed Indicator ($E$)**:
  * $E = 1$: Event observed (e.g. comment posted, issue closed, PR merged).
  * $E = 0$: Right-censored (still pending / open as of the observation snapshot).

The non-parametric Kaplan-Meier estimator computes survival step curves without assuming an underlying parametric distribution.


In [5]:
def compute_km_curve(durations: np.ndarray, events: np.ndarray, max_days: float = 90.0):
    df_km = pd.DataFrame({"duration": durations, "event": events})
    df_km = df_km[df_km["duration"] >= 0].sort_values("duration").reset_index(drop=True)
    
    unique_times = np.sort(df_km["duration"].unique())
    # Cap analysis horizon at max_days to prevent long-tail distortion
    unique_times = unique_times[unique_times <= max_days]
    
    n_at_risk = len(df_km)
    survival_prob = 1.0
    var_sum = 0.0
    
    timeline = [0.0]
    survival = [1.0]
    ci_lower = [1.0]
    ci_upper = [1.0]
    
    for t in unique_times:
        sub = df_km[df_km["duration"] == t]
        d = sub["event"].sum() # number of events at time t
        c = len(sub) - d       # censored items at time t
        
        if n_at_risk <= 0:
            break
            
        if d > 0:
            survival_prob *= (1.0 - d / n_at_risk)
            if n_at_risk > d:
                var_sum += d / (n_at_risk * (n_at_risk - d))
                
        std_err = survival_prob * np.sqrt(var_sum)
        
        timeline.append(t)
        survival.append(survival_prob)
        ci_lower.append(max(0.0, survival_prob - 1.96 * std_err))
        ci_upper.append(min(1.0, survival_prob + 1.96 * std_err))
        
        n_at_risk -= (d + c)
        
    return {
        "timeline": np.array(timeline),
        "survival": np.array(survival),
        "ci_lower": np.array(ci_lower),
        "ci_upper": np.array(ci_upper)
    }

def prepare_survival_cohorts(df_issues: pd.DataFrame, df_prs: pd.DataFrame, now: dt.datetime = None):
    if now is None:
        now = dt.datetime.now(dt.timezone.utc)
        
    # 1. Issue First Comment
    iss_resp_dur = []
    iss_resp_evt = []
    for _, row in df_issues.iterrows():
        if pd.notna(row["first_response_at"]):
            dur = (row["first_response_at"] - row["created_at"]).total_seconds() / 86400.0
            iss_resp_dur.append(dur)
            iss_resp_evt.append(1)
        else:
            dur = (now - row["created_at"]).total_seconds() / 86400.0
            iss_resp_dur.append(dur)
            iss_resp_evt.append(0)
            
    # 2. PR First Review / Comment
    pr_resp_dur = []
    pr_resp_evt = []
    for _, row in df_prs.iterrows():
        if pd.notna(row["first_review_at"]):
            dur = (row["first_review_at"] - row["created_at"]).total_seconds() / 86400.0
            pr_resp_dur.append(dur)
            pr_resp_evt.append(1)
        else:
            dur = (now - row["created_at"]).total_seconds() / 86400.0
            pr_resp_dur.append(dur)
            pr_resp_evt.append(0)
            
    # 3. Issue Close / Resolution
    iss_close_dur = []
    iss_close_evt = []
    for _, row in df_issues.iterrows():
        if pd.notna(row["closed_at"]):
            dur = (row["closed_at"] - row["created_at"]).total_seconds() / 86400.0
            iss_close_dur.append(dur)
            iss_close_evt.append(1)
        else:
            dur = (now - row["created_at"]).total_seconds() / 86400.0
            iss_close_dur.append(dur)
            iss_close_evt.append(0)
            
    # 4. PR Close / Resolution (Combined)
    pr_close_dur = []
    pr_close_evt = []
    
    # 4a. PR Merged specifically
    pr_merged_dur = []
    pr_merged_evt = []
    
    # 4b. PR Closed Unmerged
    pr_unmerged_dur = []
    pr_unmerged_evt = []
    
    for _, row in df_prs.iterrows():
        if pd.notna(row["closed_at"]):
            dur = (row["closed_at"] - row["created_at"]).total_seconds() / 86400.0
            pr_close_dur.append(dur)
            pr_close_evt.append(1)
            
            if pd.notna(row["merged_at"]):
                pr_merged_dur.append(dur)
                pr_merged_evt.append(1)
            else:
                pr_unmerged_dur.append(dur)
                pr_unmerged_evt.append(1)
        else:
            dur = (now - row["created_at"]).total_seconds() / 86400.0
            pr_close_dur.append(dur)
            pr_close_evt.append(0)
            pr_merged_dur.append(dur)
            pr_merged_evt.append(0)
            pr_unmerged_dur.append(dur)
            pr_unmerged_evt.append(0)
            
    return {
        "issue_response": (np.array(iss_resp_dur), np.array(iss_resp_evt)),
        "pr_response": (np.array(pr_resp_dur), np.array(pr_resp_evt)),
        "issue_close": (np.array(iss_close_dur), np.array(iss_close_evt)),
        "pr_close": (np.array(pr_close_dur), np.array(pr_close_evt)),
        "pr_merged": (np.array(pr_merged_dur), np.array(pr_merged_evt)),
        "pr_unmerged": (np.array(pr_unmerged_dur), np.array(pr_unmerged_evt))
    }

cohorts = prepare_survival_cohorts(df_issues, df_prs)
print("Computed durations and censoring events across all cohorts.")


Computed durations and censoring events across all cohorts.


## Visualization 1: Issue & PR Activity Survival Curves (The Whiteboard Model)

Direct implementation of Cali Dolfi's whiteboard diagram from 8Knot Issue #505:
* **Green Step Curve**: `Issue First Comment` (Survival probability of an issue remaining without external response).
* **Gray Step Curve**: `PR First Comment / Review` (Survival probability of a PR waiting for first review).
* **Red Step Curve**: `PR Closed / Resolved` (Survival probability of a PR remaining open).
* **Blue Step Curve**: `Issue Closed / Resolved` (Survival probability of an issue remaining open).
* **X-axis**: Elapsed time in days ($t = 0 	o 90$ days).
* **Y-axis**: Survival Probability $\hat{S}(t) \in [0, 1]$.


In [6]:
def plot_whiteboard_survival_curves(cohorts: dict, max_days: float = 90.0):
    curves = {
        "Issue First Comment": (cohorts["issue_response"], "#27AE60", "solid"),
        "PR First Comment / Review": (cohorts["pr_response"], "#7F8C8D", "solid"),
        "PR Closed / Resolved": (cohorts["pr_close"], "#E74C3C", "solid"),
        "Issue Closed / Resolved": (cohorts["issue_close"], "#2980B9", "solid")
    }
    
    fig = go.Figure()
    
    for label, ((dur, evt), color, dash) in curves.items():
        km = compute_km_curve(dur, evt, max_days=max_days)
        fig.add_trace(go.Scatter(
            x=km["timeline"],
            y=km["survival"],
            mode="lines",
            line=dict(color=color, width=2.5, shape="hv", dash=dash),
            name=label,
            hovertemplate=f"<b>{label}</b><br>Elapsed: %{{x:.1f}} days<br>Survival Prob: %{{y:.1%}}<extra></extra>"
        ))
        
    # Median reference line (50% survival horizon)
    fig.add_hline(
        y=0.50,
        line_dash="dot",
        line_color="#BDC3C7",
        annotation_text="Median Survival (50%)",
        annotation_position="bottom right"
    )
    
    fig.update_layout(
        title="Issue & Pull Request Activity Survival Analysis (Kaplan-Meier Curves)",
        xaxis_title="Elapsed Time Since Creation (Days)",
        yaxis_title="Survival Probability S(t)",
        yaxis=dict(range=[0, 1.05], tickformat=".0%"),
        xaxis=dict(range=[0, max_days]),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=500
    )
    return fig

fig_whiteboard = plot_whiteboard_survival_curves(cohorts)
fig_whiteboard.show()


## Visualization 2: PR Resolution Disaggregation (Merged vs Closed Unmerged)

Directly addresses Cali Dolfi's core conceptual question on the whiteboard:
> *"Should merge be separate?"*

In healthy open-source communities, pull requests that successfully merge follow an active, collaborative turnaround path, whereas pull requests that close unmerged often stall, linger, or sit abandoned before final triage. Disaggregating the curves verifies this divergence.


In [7]:
def plot_pr_merged_vs_unmerged_survival(cohorts: dict, max_days: float = 90.0):
    fig = go.Figure()
    
    km_merged = compute_km_curve(cohorts["pr_merged"][0], cohorts["pr_merged"][1], max_days=max_days)
    km_unmerged = compute_km_curve(cohorts["pr_unmerged"][0], cohorts["pr_unmerged"][1], max_days=max_days)
    
    fig.add_trace(go.Scatter(
        x=km_merged["timeline"],
        y=km_merged["survival"],
        mode="lines",
        line=dict(color="#27AE60", width=3, shape="hv"),
        name="PR Merged (Accepted)",
        hovertemplate="<b>PR Merged</b><br>Elapsed: %{x:.1f}d<br>Survival: %{y:.1%}<extra></extra>"
    ))
    
    fig.add_trace(go.Scatter(
        x=km_unmerged["timeline"],
        y=km_unmerged["survival"],
        mode="lines",
        line=dict(color="#C0392B", width=2.5, shape="hv", dash="dash"),
        name="PR Closed Unmerged (Rejected / Abandoned)",
        hovertemplate="<b>PR Closed Unmerged</b><br>Elapsed: %{x:.1f}d<br>Survival: %{y:.1%}<extra></extra>"
    ))
    
    fig.update_layout(
        title="Pull Request Survival: Merged vs Closed Unmerged Disaggregation",
        xaxis_title="Elapsed Time Since PR Open (Days)",
        yaxis_title="Survival Probability S(t)",
        yaxis=dict(range=[0, 1.05], tickformat=".0%"),
        xaxis=dict(range=[0, max_days]),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=480
    )
    return fig

fig_pr_compare = plot_pr_merged_vs_unmerged_survival(cohorts)
fig_pr_compare.show()


## Visualization 3: Empirical Turnaround Percentiles & SLA Horizons

Summarizes the survival distributions into operational milestones:
* **25% Resolved**: Early responder benchmark (first quartile turnaround).
* **50% Resolved**: Median project turnaround.
* **75% Resolved**: Upper bound turnaround for majority of contributions.
* **90% Resolved**: Tail latency identifying outlier items requiring triage.


In [8]:
def compute_quantiles_table(cohorts: dict):
    labels = [
        ("Issue First Comment", cohorts["issue_response"]),
        ("PR First Comment / Review", cohorts["pr_response"]),
        ("PR Merged", cohorts["pr_merged"]),
        ("Issue Closed", cohorts["issue_close"])
    ]
    
    data = []
    for name, (dur, evt) in labels:
        km = compute_km_curve(dur, evt, max_days=180.0)
        
        def find_time_at_prob(target_p):
            idx = np.where(km["survival"] <= target_p)[0]
            if len(idx) > 0:
                return round(km["timeline"][idx[0]], 1)
            return np.nan
            
        p25 = find_time_at_prob(0.75) # 25% resolved
        p50 = find_time_at_prob(0.50) # median
        p75 = find_time_at_prob(0.25) # 75% resolved
        p90 = find_time_at_prob(0.10) # 90% resolved
        
        data.append({
            "Activity Stream": name,
            "25% Resolved (Days)": p25,
            "50% Median (Days)": p50,
            "75% Resolved (Days)": p75,
            "90% Resolved (Days)": p90
        })
        
    df_q = pd.DataFrame(data)
    return df_q

df_quantiles = compute_quantiles_table(cohorts)

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{col}</b>" for col in df_quantiles.columns],
        fill_color="#2C3E50",
        font=dict(color="white", size=12),
        align="left"
    ),
    cells=dict(
        values=[df_quantiles[col] for col in df_quantiles.columns],
        fill_color="#F8F9FA",
        font=dict(color="#2C3E50", size=12),
        align="left",
        height=30
    )
)])

fig_table.update_layout(
    title="Activity Turnaround Milestones (Kaplan-Meier Empirical Quantiles)",
    margin=dict(l=40, r=40, t=60, b=30),
    height=260
)
fig_table.show()


## Visualization 4: Current Open Inventory & Right-Censoring Profile

Highlights active work items currently open in the community by age bracket (<7d, 7-30d, 30-90d, 90d+).
Unresolved items in the older age brackets represent right-censored items that risk stalling without maintainer intervention.


In [9]:
def plot_open_inventory_age_profile(df_issues: pd.DataFrame, df_prs: pd.DataFrame):
    now = dt.datetime.now(dt.timezone.utc)
    
    open_issues = df_issues[df_issues["closed_at"].isna()].copy()
    open_prs = df_prs[df_prs["closed_at"].isna()].copy()
    
    open_issues["age_days"] = (now - open_issues["created_at"]).dt.total_seconds() / 86400.0
    open_prs["age_days"] = (now - open_prs["created_at"]).dt.total_seconds() / 86400.0
    
    def bucket_age(days):
        if days < 7:
            return "Fresh (< 7d)"
        elif days <= 30:
            return "Active (7-30d)"
        elif days <= 90:
            return "Aging (30-90d)"
        else:
            return "Stale (90d+)"
            
    open_issues["age_bucket"] = open_issues["age_days"].apply(bucket_age)
    open_prs["age_bucket"] = open_prs["age_days"].apply(bucket_age)
    
    order = ["Fresh (< 7d)", "Active (7-30d)", "Aging (30-90d)", "Stale (90d+)"]
    
    iss_counts = open_issues["age_bucket"].value_counts().reindex(order, fill_value=0)
    pr_counts = open_prs["age_bucket"].value_counts().reindex(order, fill_value=0)
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name="Open Issues",
        x=order,
        y=iss_counts.values,
        marker_color="#2980B9",
        text=iss_counts.values,
        textposition="outside"
    ))
    
    fig.add_trace(go.Bar(
        name="Open Pull Requests",
        x=order,
        y=pr_counts.values,
        marker_color="#8E44AD",
        text=pr_counts.values,
        textposition="outside"
    ))
    
    fig.update_layout(
        title="Open Work Item Inventory by Age Bracket (Right-Censored Cohorts)",
        xaxis_title="Age Bracket",
        yaxis_title="Count of Open Items",
        barmode="group",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=460
    )
    return fig

fig_inventory = plot_open_inventory_age_profile(df_issues, df_prs)
fig_inventory.show()


## 8Knot Visualization Specification & Implementation Guide

This prototype establishes the analytical foundation for the 8Knot issue & PR survival visualization (`8Knot/pages/contributions/visualizations/issue_pr_survival.py`):
* **Data Sources**: Seamlessly connects to existing `issues_query`, `prs_query`, and `pr_response_query` without database modifications.
* **Non-Author Comment Identification**:
  * Resolves the bug encountered in 8Knot PR #741 where the issue first comment line was stuck at zero by filtering:
    `msg_timestamp > created_at + 5s` and `cntrb_id[:15] != msg_cntrb_id[:15]`.
* **Interactive Controls for Dash**:
  1. `Checklist`: Toggle individual survival curves (`Issue First Comment`, `PR First Comment`, `Issue Closed`, `PR Closed`, `PR Merged`).
  2. `Slider`: Adjust survival observation horizon (30d, 60d, 90d, 180d, 365d).
  3. `Dropdown`: Filter by repository or repository group.
